In [9]:
import construction_helpers as ch # Needed for @ auto-alignment
import data_structure.Category as cat
import data_structure.Numeric as nm
import data_structure.Operators as ops
import data_structure.Term as fd
import display as dpl
import websocket_transfer.websockets_transfer as wst

In [10]:
mh_attention = cat.Block.template(
      ops.Einops.template('q h d, x h d -> h q x')
    @ ops.SoftMax.template()
    @ ops.WeightedTriangularLower.template()
    @ ops.Einops.template('h q x, x h d -> q h d'),
    title='Attention Core',
    fill_color='#C5BEDF'
)
attention_block = cat.Block.template(
    (0, 0, 0) 
    @(ops.Linear.template('m', 2, 'q')
    * ops.Linear.template('m', 2, 'k')
    * ops.Linear.template('m', 2, 'v'))
    @ mh_attention
    @ ops.Linear.template(2, 'm'),
    title = 'Multi-Head Attention', fill_color='#FFE2BB'
)
ffn = cat.Block.template(
    ops.Linear.template(1, ('d_ff',), 'in') 
    @ ops.Elementwise.template() 
    @ ops.Linear.template(('d_ff',), 1, 'out'),
    title='Feed Forward', fill_color='#C1E8F7'
)
embedding = cat.Block.template(
    ops.Embedding.template('v'),
    title='Embedding', fill_color='#FCE0E1'
)
aggregator = cat.Block.template(
    ops.Linear.template(1, 'v')
    @ ops.SoftMax.template(),
    title='Aggregator', fill_color='#DBDFEF'
)
def resnet(target):
    return cat.Block.template(
          (0, 0)
        @ target
        @ ops.AdditionOp.template()
        @ ops.Normalize.template(),
        title='Add \\& Norm', fill_color='#F1F4C1'
    )
transformer = (
    embedding
    @ cat.Block.template(
        resnet(attention_block)
        @ resnet(ffn),
        repetition=6,
        title='Transformer Layer', fill_color='#F3F3F4'
    )
    @ aggregator
)
await wst.send_term(transformer)


Received from server: {"msgType": "Connected"}
Received from server: {"msgType": "DataReceived"}
